In [3]:
#대서산업 TeamViewer로 부터 이미지 폴더를 복사하여 붙여넣기 한 후, 폴더 내 파일명을 리스트업하는 코드
from pathlib import Path

folder_path = Path(r"C:\Users\User\Desktop\Project\VISION_AI")
src_folder = folder_path / "20260428"
files = [f for f in src_folder.iterdir() if f.is_file() and 'front' in f.name]

print(len(files))

148


In [4]:
#대서산업 이미지에서 front가 포함된 파일명만 리스트업하여 dataset 폴더(roboflow 업로드용)로 복사하는 코드
import shutil
# 1. 경로 설정
dest_folder = Path("./dataset") # 저장할 폴더
dest_folder.mkdir(parents=True, exist_ok=True) # 폴더가 없으면 생성

# 2. 파일 복사 작업
for file_path in files:
    target_path = dest_folder / file_path.name
    # 파일 복사
    shutil.copy2(file_path, target_path) # copy2는 메타데이터(수정시간 등)도 보존함
    print(f"COPIED: {file_path.name}")

print("--- 작업 완료 ---")

COPIED: front_1777329632545_대구06라5245.BMP
COPIED: front_1777330189100_대구06라5653.BMP
COPIED: front_1777330238391_006나7520.BMP
COPIED: front_1777330285211_대구06라5248.BMP
COPIED: front_1777330327701_대구06라5245.jpg
COPIED: front_1777330329119_대구06라5226.BMP
COPIED: front_1777330329408_대구06라5245.jpg
COPIED: front_1777330372795_006다6192.BMP
COPIED: front_1777330459818_대구06라5653.jpg
COPIED: front_1777330570191_006나7520.jpg
COPIED: front_1777330710366_대구06라5226.jpg
COPIED: front_1777330712038_대구06라5226.jpg
COPIED: front_1777330841412_대구06라5248.jpg
COPIED: front_1777330844195_대구06라5248.jpg
COPIED: front_1777331015337_006다6192.jpg
COPIED: front_1777331419455_대구06다7763.BMP
COPIED: front_1777331666118_대구06다7763.jpg
COPIED: front_1777332613894_006마8899.BMP
COPIED: front_1777332893736_006마8899.jpg
COPIED: front_1777332994657_대구06라5245.BMP
COPIED: front_1777333037946_006나7520.BMP
COPIED: front_1777333070873_대구06라5653.BMP
COPIED: front_1777333109536_대구06라5226.BMP
COPIED: front_1777333292759_대구06라5245.jpg

In [6]:
#대서산업 이미지를 roboflow에 업로드하는 코드
import roboflow

rf = roboflow.Roboflow(api_key="Hci8cXDA4hpgvUQwnPFz")

# List all projects for your workspace
workspace = rf.workspace()

# get a specific project
project = rf.workspace().project("20260428")

# list all versions in a specific project
# project.versions()

# Upload data set to a new/existing project
workspace.upload_dataset(
    "./dataset/", # This is your dataset path
    "20260428", # This will either create or get a dataset with the given ID
    num_workers=10,
    project_license="MIT",
    project_type="object-detection",
    batch_name=None,
    num_retries=0,
    is_prediction=False #optional, set to True if the dataset is not ground truth and needs approval
)

loading Roboflow workspace...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow project...
Uploading to existing project jongkwons-workspace/20260428


100%|██████████| 148/148 [00:00<?, ?it/s]


[DUPLICATE] ./dataset/front_1777330570191_006나7520.jpg (5wKHZsaq73EqdoYCtPJb) [2.8s]
[DUPLICATE] ./dataset/front_1777330459818_대구06라5653.jpg (WpGmJFcGKfc0q10OPIUX) [3.1s]
[DUPLICATE] ./dataset/front_1777330329408_대구06라5245.jpg (ucHhhKcnRxkiiFuUyIH7) [3.2s]
[DUPLICATE] ./dataset/front_1777330327701_대구06라5245.jpg (ucHhhKcnRxkiiFuUyIH7) [3.2s]
[DUPLICATE] ./dataset/front_1777330710366_대구06라5226.jpg (k7QzYRCk6g5AXyyI8zwQ) [1.9s]
[DUPLICATE] ./dataset/front_1777330712038_대구06라5226.jpg (k7QzYRCk6g5AXyyI8zwQ) [2.1s]
[DUPLICATE] ./dataset/front_1777330841412_대구06라5248.jpg (YpySTzuPfleIEAFgpvUF) [2.2s]
[DUPLICATE] ./dataset/front_1777330844195_대구06라5248.jpg (YpySTzuPfleIEAFgpvUF) [2.2s]
[DUPLICATE] ./dataset/front_1777331015337_006다6192.jpg (povFeHMxMQQ2zkaa67vd) [1.8s]
[DUPLICATE] ./dataset/front_1777330329119_대구06라5226.BMP (uTPiSDToLNp3Ck487pBv) [6.9s]
[DUPLICATE] ./dataset/front_1777330372795_006다6192.BMP (h11GnPYuaFvVIzGqXdCF) [7.0s]
[DUPLICATE] ./dataset/front_1777330285211_대구06라5248.BMP (

In [7]:
rf = roboflow.Roboflow(api_key="Hci8cXDA4hpgvUQwnPFz")
project = rf.workspace("jongkwons-workspace").project("20260428")
project.versions()

loading Roboflow workspace...
loading Roboflow project...


In [8]:
dataset = project.version(3).download("yolo26")


Extracting Dataset Version Zip to 20260428-3 in yolo26:: 100%|██████████| 259/259 [00:00<00:00, 1110.57it/s]


In [ ]:
import os
import re
import shutil

# 1. 경로 설정
BASE_DIR = r'c:\Users\User\Desktop\Project\VISION_AI\20260428-3'
ORIGINAL_SOURCE_DIR = r'c:\Users\User\Desktop\Project\VISION_AI\20260428'  # 원본 파일들이 있는 곳
SUB_SETS = ['train', 'valid', 'test']

def get_original_info(rf_filename):
    """
    Roboflow 파일명에서 원본 파일명과 확장자를 복원합니다.
    """
    # .rf.해시값... 제거
    clean = re.sub(r'\.rf\.[a-z0-9]+', '', rf_filename, flags=re.IGNORECASE)
    
    # 확장자 복원 및 원본 확장자 식별
    actual_name = clean
    if '_BMP' in clean:
        actual_name = clean.replace('_BMP', '.BMP')
    elif '_jpg' in clean:
        actual_name = clean.replace('_jpg', '.jpg')
    
    # 확장자를 제외한 순수 파일명 (라벨 매칭용)
    pure_name = os.path.splitext(actual_name)[0]
    
    return actual_name, pure_name

def process_replacement(base_path):
    for subset in SUB_SETS:
        subset_path = os.path.join(base_path, subset)
        if not os.path.exists(subset_path): continue
            
        print(f"\n--- [{subset.upper()}] 원본 교체 및 라벨 정리 시작 ---")
        
        img_dir = os.path.join(subset_path, 'images')
        lbl_dir = os.path.join(subset_path, 'labels')
        
        if not os.path.exists(img_dir): continue

        img_files = os.listdir(img_dir)
        success_count = 0

        for f in img_files:
            old_img_path = os.path.join(img_dir, f)
            if not os.path.isfile(old_img_path): continue

            # 1. 정보 추출
            original_filename, pure_name = get_original_info(f)
            source_file_path = os.path.join(ORIGINAL_SOURCE_DIR, original_filename)

            # 2. 이미지 교체 (Source -> Destination)
            if os.path.exists(source_file_path):
                # 기존 Roboflow 이미지 삭제
                os.remove(old_img_path)
                # 원본 소스에서 새 이미지 복사
                shutil.copy2(source_file_path, os.path.join(img_dir, original_filename))
                
                # 3. 라벨 파일 이름 변경 (이미지 이름과 일치시키기)
                # 기존 라벨 파일 찾기 (파일명에 해시가 포함되어 있으므로 glob 패턴처럼 검색)
                rf_label_name = f.replace(os.path.splitext(f)[1], '.txt')
                old_lbl_path = os.path.join(lbl_dir, rf_label_name)
                new_lbl_path = os.path.join(lbl_dir, pure_name + '.txt')

                if os.path.exists(old_lbl_path):
                    if old_lbl_path != new_lbl_path:
                        if os.path.exists(new_lbl_path): os.remove(new_lbl_path)
                        os.rename(old_lbl_path, new_lbl_path)
                
                success_count += 1
            else:
                print(f"  ⚠️ 원본 소스 없음: {original_filename}")

        print(f"  ✅ {subset} 완료: {success_count}개 파일 교체됨")

if __name__ == "__main__":
    process_replacement(BASE_DIR)
    print("\n✨ 모든 이미지가 원본 소스로 대체되었고 라벨 이름이 동기화되었습니다!")

In [12]:
dataset.location.split("\\")[-1]

'20260428-3'

In [13]:
dataset.location

'c:\\Users\\User\\Desktop\\Project\\VISION_AI\\20260428-3'

In [9]:
def ftp_makedirs(ftp, remote_path):
    """서버에 중첩된 폴더 구조를 자동으로 생성하는 함수"""
    path_parts = [p for p in remote_path.split('/') if p]
    current_path = ""
    for part in path_parts:
        current_path += f"/{part}"
        try:
            ftp.cwd(current_path)
        except:
            ftp.mkd(current_path)
            ftp.cwd(current_path)

In [ ]:
import os
from ftplib import FTP
from tqdm import tqdm

# 1. 설정
FTP_HOST = "192.168.1.179"
FTP_USER = "administrator"
FTP_PASS = "waff!23"
REMOTE_FOLDER = "/ai_vision/yolo"
LOCAL_FOLDER = dataset.location

# 2. FTP 연결
ftp = FTP()
ftp.connect(FTP_HOST, 21)
ftp.login(user=FTP_USER, passwd=FTP_PASS)
ftp.encoding = "utf-8"  # 한글 깨짐 방지

try:
    # 모든 파일 목록 먼저 수집 (tqdm 전체 개수 파악용)
    all_files = []
    for root, dirs, files in os.walk(LOCAL_FOLDER):
        for file in files:
            all_files.append(os.path.join(root, file))

    print(f"총 {len(all_files)}개의 파일을 업로드합니다.")

    # 3. 폴더 순회 및 업로드
    for local_file in tqdm(all_files, desc="전체 폴더 업로드 중", unit="file"):
        # 로컬 상대 경로를 추출하여 서버 경로 생성
        rel_path = os.path.relpath(local_file, LOCAL_FOLDER)
        remote_file_path = os.path.join(REMOTE_FOLDER, rel_path).replace("\\", "/")
        remote_dir = os.path.dirname(remote_file_path)

        # 서버에 해당 폴더가 있는지 확인 및 생성
        ftp_makedirs(ftp, remote_dir)

        # 파일 전송
        with open(local_file, "rb") as f:
            ftp.storbinary(f"STOR {os.path.basename(remote_file_path)}", f)

    print("\n--- 폴더 전체 복사 완료 ---")

except Exception as e:
    print(f"\n오류 발생: {e}")

finally:
    ftp.quit()

# 2. 환영 메시지 및 현재 파일 목록 출력
#print(ftp.getwelcome())
#ftp.dir() 

# 2. 파일 목록 조회
# nlst(): 파일/폴더 이름만 리스트로 반환 (프로그래밍하기 편함)
#files = ftp.nlst()

총 257개의 파일을 업로드합니다.


전체 폴더 업로드 중: 100%|██████████| 257/257 [00:35<00:00,  7.17file/s]


--- 폴더 전체 복사 완료 ---


In [70]:
import paramiko

# --- 서버 접속 정보 ---
HOST = "192.168.1.179"
USER = "administrator"
PASS = "waff!23"

# 서버에 이미 존재하는 파일의 전체 경로
# 1. 가상환경 내 파이썬 경로
VENV_PYTHON = r'D:\ai_vision\yolo\venv\Scripts\python.exe'
# 2. 이동할 작업 디렉토리
WORKING_DIR = r'D:\ai_vision\yolo'
# 3. 실행할 스크립트 파일 이름 (이미 해당 폴더 안이라면 파일명만 써도 됨)
SCRIPT_NAME = 'train.py'
def test_ssh_execution():
    print(f"[{HOST}] 연결 시도 중...")
    
    ssh = paramiko.SSHClient()
    # 서버의 호스트 키가 없어도 자동으로 등록하도록 설정
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    
    try:
        # 1. SSH 접속
        ssh.connect(HOST, username=USER, password=PASS)
        print("✅ SSH 접속 성공!")

        # 2. 명령어 실행 (PowerShell을 통해 가상환경 실행)
        command = f'powershell.exe -Command "Set-Location \'{WORKING_DIR}\'; & \'{VENV_PYTHON}\' \'{SCRIPT_NAME}\'"'
        print(f"실행 명령어: {command}")
        # 2. 명령어 실행 (PowerShell을 통해 Python 실행)
        #command = f'powershell.exe -Command "python {REMOTE_FILE_PATH}"'
        #print(f"실행 명령어: {command}")
        
        stdin, stdout, stderr = ssh.exec_command(command)

        # 3. 결과 읽기 (윈도우 서버 결과는 cp949 인코딩이 많음)
        output = stdout.read().decode('cp949', errors='ignore')
        error = stderr.read().decode('cp949', errors='ignore')

        if output:
            print("\n--- [실행 결과] ---")
            print(output)
        
        if error:
            print("\n--- [에러 메시지] ---")
            print(error)
            print("💡 팁: 서버에 Python이 설치되어 있고 환경변수(Path)에 등록되어 있는지 확인하세요.")

    except Exception as e:
        print(f"❌ 접속 또는 실행 중 오류 발생: {e}")
        
    finally:
        ssh.close()
        print("\nSSH 연결 종료.")

if __name__ == "__main__":
    test_ssh_execution()

[192.168.1.179] 연결 시도 중...
✅ SSH 접속 성공!
실행 명령어: powershell.exe -Command "Set-Location 'D:\ai_vision\yolo'; & 'D:\ai_vision\yolo\venv\Scripts\python.exe' 'train.py'"

SSH 연결 종료.


KeyboardInterrupt: 